# Privacy Demo
This notebook demonstrates simple privacy-preserving techniques (e.g., masking, aggregation, differential privacy examples) on sample data.

Sections:
- Sample data generation
- Masking and redaction
- Aggregation and k-anonymity checks
- (Optional) Differential privacy sketch

In [2]:
# Placeholder: import libraries
import pandas as pd

# TODO: add privacy-preserving examples

# Privacy Risk Assessment — NovaCred Credit Applications

## Context

NovaCred is a fintech company that uses data-driven models to evaluate loan applications and assess applicants’ creditworthiness. Because these systems process large amounts of personal and financial data and may produce automated decisions affecting individuals’ access to credit, strong data governance and privacy protections are required.

This notebook performs a privacy risk assessment of the NovaCred credit application dataset. The objective is to identify potential privacy risks, assess the presence of personally identifiable information (PII), and evaluate whether the dataset aligns with key regulatory frameworks such as the General Data Protection Regulation (GDPR) and the EU AI Act.

The analysis focuses on:

• Identifying personally identifiable information (PII)  
• Evaluating potential re-identification risks  
• Demonstrating pseudonymization techniques  
• Mapping dataset attributes to relevant GDPR principles  
• Assessing whether the system may qualify as a high-risk AI system under the EU AI Act

The goal is to demonstrate how privacy-preserving practices and governance controls can be integrated into a credit decision data pipeline to support responsible and compliant AI deployment.

## 1. Dataset Loading

The dataset used in this assessment contains simulated credit application records for the NovaCred system.

In this step, the raw JSON dataset is loaded and converted into a structured pandas DataFrame. This allows us to inspect the attributes present in the dataset and identify fields that may contain personal or sensitive information.

Understanding the structure of the dataset is an essential first step before performing privacy risk analysis, PII identification, and regulatory mapping.

In [3]:
import pandas as pd
import numpy as np
import json

In [4]:
# Load raw dataset
with open("../data/data.json", "r") as f:
    data = json.load(f)

print("Number of credit applications:", len(data))

Number of credit applications: 502


In [5]:
df = pd.json_normalize(data)

df.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN


In [14]:
df.columns

Index(['_id', 'spending_behavior', 'processing_timestamp',
       'applicant_info.full_name', 'applicant_info.email',
       'applicant_info.ssn', 'applicant_info.ip_address',
       'applicant_info.gender', 'applicant_info.date_of_birth',
       'applicant_info.zip_code', 'financials.annual_income',
       'financials.credit_history_months', 'financials.debt_to_income',
       'financials.savings_balance', 'decision.loan_approved',
       'decision.rejection_reason', 'loan_purpose', 'decision.interest_rate',
       'decision.approved_amount', 'financials.annual_salary', 'notes',
       'email_hash'],
      dtype='object')

In [15]:
print("Total attributes in dataset:", len(df.columns))

Total attributes in dataset: 22


## 2. PII Identification

To evaluate privacy risks, we first identify fields that contain personally identifiable information (PII) or personal data as defined under **GDPR Article 4(1)**.

These attributes may allow direct identification of individuals or increase the risk of re-identification when combined with other attributes in the dataset.

### 2.1 PII Fields in the Dataset

The dataset contains several attributes that can directly or indirectly identify individuals.

These fields represent potential privacy risks and must be carefully handled in data governance and regulatory compliance contexts.

In [7]:
pii_fields = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.gender",
    "applicant_info.date_of_birth",
    "applicant_info.zip_code"
]

print("Potential PII fields identified:\n")

for field in pii_fields:
    print("-", field)

Potential PII fields identified:

- applicant_info.full_name
- applicant_info.email
- applicant_info.ssn
- applicant_info.ip_address
- applicant_info.gender
- applicant_info.date_of_birth
- applicant_info.zip_code


### 2.2 PII Risk Classification

After identifying potential PII fields, we classify them based on their **identification risk** and **privacy sensitivity**.

We distinguish between:

- **Direct identifiers** – uniquely identify an individual (e.g., name, SSN)
- **Quasi-identifiers** – may identify individuals when combined with other attributes
- **Sensitive attributes** – data that may reveal personal behavior or protected characteristics

In [17]:
print("Total attributes in dataset:", len(df.columns))
print("Potential PII attributes identified:", len(pii_classification))

pii_classification = {
    "applicant_info.full_name": "Direct identifier",
    "applicant_info.email": "Direct identifier",
    "applicant_info.ssn": "Highly sensitive identifier",
    "applicant_info.ip_address": "Online identifier (GDPR Art.4)",
    "applicant_info.date_of_birth": "Quasi identifier",
    "applicant_info.zip_code": "Quasi identifier",
    "applicant_info.gender": "Protected attribute",
    "spending_behavior": "Sensitive behavioral data",
    "notes": "Unstructured personal data (potential PII)"
}

pd.DataFrame.from_dict(
    pii_classification,
    orient="index",
    columns=["PII Category"]
)

Total attributes in dataset: 22
Potential PII attributes identified: 9


,PII Category
applicant_info.full_name,Direct identifier
applicant_info.email,Direct identifier
applicant_info.ssn,Highly sensitive identifier
applicant_info.ip_address,Online identifier (GDPR Art.4)
applicant_info.date_of_birth,Quasi identifier
applicant_info.zip_code,Quasi identifier
applicant_info.gender,Protected attribute
spending_behavior,Sensitive behavioral data
notes,Unstructured personal data (potential PII)


These classifications highlight the privacy risks associated with different attributes in the dataset and support a structured privacy risk assessment of the NovaCred data pipeline in line with GDPR principles.

Direct identifiers such as names, email addresses, and social security numbers pose the highest identification risk because they uniquely identify an individual without requiring additional information. If exposed in a data breach, these attributes would immediately reveal the identity of applicants.

Quasi-identifiers such as ZIP code and date of birth may not uniquely identify individuals on their own, but they can enable re-identification when combined with other attributes. Research has shown that combinations of seemingly harmless attributes can significantly increase identification risk, particularly when linked with external datasets.

Online identifiers such as IP addresses are also considered personal data under GDPR Article 4 because they can be used to single out or track individuals when combined with system logs or network metadata.

Behavioral attributes such as `spending_behavior` can reveal sensitive lifestyle and financial patterns. Even though these attributes do not directly identify an individual, they may enable inference of personal characteristics such as financial stress, health-related spending, or personal habits.

Unstructured text fields such as `notes` represent an additional privacy risk because free-text data may contain personal information that is not explicitly defined in the schema. Such fields can unintentionally include names, contact details, or contextual information that could enable identification.

From a governance perspective, this classification helps determine which attributes require stronger privacy protections such as pseudonymization, restricted access controls, or data minimization before the dataset can be used in analytical or machine learning pipelines.

## 3. Pseudonymization

To reduce privacy risks, direct identifiers can be replaced with pseudonymous values. 
Pseudonymization allows data to be used for analysis while protecting the original identity of individuals.

In this example, we apply hashing to email addresses to replace the original identifier with a pseudonymous value.

In [ ]:
import hashlib
import pandas as pd

# Function to pseudonymize email using SHA-256 hashing
def hash_email(email):
    if pd.isna(email):
        return email
    return hashlib.sha256(email.encode()).hexdigest()

# Create pseudonymized column
df["email_hash"] = df["applicant_info.email"].apply(hash_email)

# Show original vs pseudonymized values
print("Sample of original vs pseudonymized email values:\n")
display(df[["applicant_info.email", "email_hash"]].head())

# Data minimization: remove original identifier
df_privacy = df.drop(columns=["applicant_info.email"])

print("\nOriginal column count:", df.shape[1])
print("Column count after removing email:", df_privacy.shape[1])
print("Original email column removed for privacy protection.")

Sample of original vs pseudonymized email values:



,applicant_info.email,email_hash
0,jerry.smith17@hotmail.com,116648a7761525746032d0ab323ceb01f50d11f7935164...
1,brandon.walker2@yahoo.com,c3522c0b54ef9045c73186bcabb53f8e512360ed17e9cc...
2,scott.moore94@mail.com,b299e7d6a37e183bab209eb8df919652117dd16ed16698...
3,thomas.lee6@protonmail.com,6fbd2478748a29faa143392f28d955133e346d09673963...
4,brian.rodriguez86@aol.com,f24e7cc1450ee9aa26b833e0593b2420fbfd59b8ad2636...



Original column count: 22
Column count after removing email: 21
Original email column removed for privacy protection.


The pseudonymization process replaces direct identifiers with hashed values, reducing the risk of exposing personal information.

Instead of storing the original email address, a cryptographic hash (SHA-256) is generated and used as a pseudonymous identifier. This allows records to be referenced consistently without revealing the underlying identity of the individual.

Pseudonymization does not fully anonymize the data, because the original identity could still be recovered if the mapping or source data were available. However, it significantly reduces the risk of direct identification in analytical workflows.

After generating the hashed identifier, the original email column is removed from the working dataset. This step demonstrates the GDPR principle of **data minimization** (Article 5(1)(c)), ensuring that only the minimum necessary personal data is retained for analysis.

In practice, a production system would typically store any identifier mappings in a separate, access-controlled identity store to further reduce re-identification risk. The analytical dataset would only contain pseudonymous identifiers.

## 4. GDPR Mapping

The identified dataset attributes and processing steps can be mapped to relevant GDPR principles and regulatory requirements. This helps clarify which legal obligations apply to the credit scoring system.

In [21]:
gdpr_mapping = {

    "Art. 6 – Lawful Basis":
    "Credit scoring must rely on a lawful basis such as contractual necessity when processing loan applications.",

    "Art. 5 – Data Minimization":
    "Only necessary attributes should be processed. Direct identifiers such as applicant_info.email should be removed after analysis.",

    "Art. 4(1) – Personal Data":
    "Attributes like applicant_info.full_name, applicant_info.email, IP address, ZIP code, and date_of_birth qualify as personal data.",

    "Art. 4(5) – Pseudonymization":
    "Email addresses are pseudonymized using SHA-256 hashing and the original identifier is removed from the analytical dataset.",

    "Art. 5(1)(f) – Integrity and Confidentiality":
    "Personal data must be processed securely using appropriate safeguards to prevent unauthorized access or disclosure.",

    "Art. 22 – Automated Decision-Making":
    "Credit scoring may involve automated decisions with significant effects, requiring transparency and safeguards.",

    "Art. 32 – Security of Processing":
    "Sensitive identifiers such as SSN require strong protection through hashing, encryption, and access control."

}

pd.DataFrame(
    list(gdpr_mapping.items()),
    columns=["GDPR Article", "Application in the Dataset"]
).style.hide(axis="index")

GDPR Article,Application in the Dataset
Art. 6 – Lawful Basis,Credit scoring must rely on a lawful basis such as contractual necessity when processing loan applications.
Art. 5 – Data Minimization,Only necessary attributes should be processed. Direct identifiers such as applicant_info.email should be removed after analysis.
Art. 4(1) – Personal Data,"Attributes like applicant_info.full_name, applicant_info.email, IP address, ZIP code, and date_of_birth qualify as personal data."
Art. 4(5) – Pseudonymization,Email addresses are pseudonymized using SHA-256 hashing and the original identifier is removed from the analytical dataset.
Art. 5(1)(f) – Integrity and Confidentiality,Personal data must be processed securely using appropriate safeguards to prevent unauthorized access or disclosure.
Art. 22 – Automated Decision-Making,"Credit scoring may involve automated decisions with significant effects, requiring transparency and safeguards."
Art. 32 – Security of Processing,"Sensitive identifiers such as SSN require strong protection through hashing, encryption, and access control."


This mapping demonstrates how the attributes in the credit application dataset relate to key GDPR provisions.

Several variables qualify as personal data under Article 4(1), including direct identifiers such as name, email address, and social security number, as well as quasi-identifiers like ZIP code and date of birth that could enable re-identification when combined with other attributes.

To reduce privacy risks, pseudonymization and data minimization techniques were applied. Direct identifiers such as email addresses were replaced with hashed values and removed from the analytical dataset after pseudonymization. This supports compliance with Article 5(1)(c) on data minimization by ensuring that only the minimum necessary personal data is retained for analytical processing.

The dataset also contains attributes used in automated credit decision processes. Under Article 22 of the GDPR, individuals have the right not to be subject solely to automated decisions that produce significant legal or similarly significant effects. Therefore, credit scoring systems must include safeguards such as transparency about automated processing, explainability of model decisions, and the possibility of human review.

Finally, strong security measures are required to protect sensitive identifiers such as social security numbers and financial information. Article 32 requires appropriate technical and organizational safeguards, including encryption, access control, and monitoring of data processing activities.

In addition, Article 5(1)(f) establishes the principle of integrity and confidentiality, requiring that personal data be processed in a way that ensures appropriate security and protection against unauthorized access, accidental loss, or disclosure.

## 5. EU AI Act — High-Risk Classification

The NovaCred credit scoring system may fall under the category of **high-risk AI systems** according to the EU AI Act. 

AI systems used to evaluate creditworthiness or determine access to essential services such as loans can significantly affect individuals' financial opportunities. Because of these potential impacts, such systems are typically classified as **high-risk**.

High-risk AI systems must comply with strict governance and risk management requirements, including transparency, documentation, human oversight, and data governance controls.

In [23]:
ai_act_mapping = {
    "AI Use Case": "Credit scoring / creditworthiness evaluation",
    "Risk Category": "High-Risk AI System",
    "Legal Reference": "EU AI Act Annex III, Point 5(b)",
    "Relevant AI Act Area": "Access to essential private services (credit)",
    "Potential Impact": "Automated decisions may affect individuals' access to financial services and economic opportunities",
    "Key Governance Requirements": "Risk management, data governance, transparency, logging, and human oversight"
}

pd.DataFrame(
    list(ai_act_mapping.items()),
    columns=["AI Act Aspect", "Assessment"]
).style.hide(axis="index")

AI Act Aspect,Assessment
AI Use Case,Credit scoring / creditworthiness evaluation
Risk Category,High-Risk AI System
Legal Reference,"EU AI Act Annex III, Point 5(b)"
Relevant AI Act Area,Access to essential private services (credit)
Potential Impact,Automated decisions may affect individuals' access to financial services and economic opportunities
Key Governance Requirements,"Risk management, data governance, transparency, logging, and human oversight"


The NovaCred credit scoring system can be classified as a **high-risk AI system** under the EU AI Act.

According to **Annex III, point 5(b)** of the EU AI Act, AI systems used to evaluate the creditworthiness of individuals or determine access to essential private services such as loans fall into the high-risk category. Because this system influences whether an applicant is approved for credit, automated decisions may have significant financial consequences for individuals.

As a result, the system must comply with strict governance and risk management requirements. These include transparency about automated decision processes, proper documentation of the system design, and mechanisms for human oversight.

In addition, the quality and fairness of the underlying data must be continuously evaluated. The dataset used for this project includes attributes such as gender, financial indicators, and behavioral spending categories, which could potentially introduce bias into automated decision-making.

To address this risk, bias analysis was performed earlier in the project (see **02-bias-analysis.ipynb**). This helps ensure that the model does not disproportionately disadvantage specific demographic groups and supports responsible deployment of the credit scoring system.

Overall, classifying the system as high-risk highlights the need for strong data governance, fairness monitoring, and human oversight throughout the lifecycle of the AI system.